# 🚗 License Plate Detector — YOLOv8 Training
**Runtime → Change runtime type → T4 GPU** before running.

## 1 · Install dependencies

In [ ]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

## 2 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3 · Configure paths

Set `DATASET_DIR` to the folder in your Drive that contains your dataset.

Expected structure inside that folder:
```
dataset/
  images/
    train/   ← training images
    val/     ← validation images
  labels/
    train/   ← YOLO .txt labels
    val/
```
If your dataset came with a `data.yaml` already, set `USE_EXISTING_YAML = True`.

In [ ]:
import os

# ── EDIT THESE ───────────────────────────────────────────────────────────────
DATASET_DIR      = '/content/drive/MyDrive/YOUR_DATASET_FOLDER'  # ← change this
USE_EXISTING_YAML = False   # True if data.yaml already exists inside DATASET_DIR
# ─────────────────────────────────────────────────────────────────────────────

YAML_PATH = os.path.join(DATASET_DIR, 'data.yaml')
print('Dataset dir :', DATASET_DIR)
print('YAML path   :', YAML_PATH)
print('Drive contents:')
!ls "{DATASET_DIR}"

## 4 · Create `data.yaml` (skip if `USE_EXISTING_YAML = True`)

In [ ]:
if not USE_EXISTING_YAML:
    yaml_content = f"""path: {DATASET_DIR}
train: images/train
val:   images/val

nc: 1
names: ['license_plate']
"""
    with open(YAML_PATH, 'w') as f:
        f.write(yaml_content)
    print('data.yaml created:')
    print(yaml_content)
else:
    print('Using existing data.yaml')
    !cat "{YAML_PATH}"

## 5 · Verify dataset

In [ ]:
import glob

train_imgs = glob.glob(os.path.join(DATASET_DIR, 'images/train/*'))
val_imgs   = glob.glob(os.path.join(DATASET_DIR, 'images/val/*'))
train_lbls = glob.glob(os.path.join(DATASET_DIR, 'labels/train/*'))
val_lbls   = glob.glob(os.path.join(DATASET_DIR, 'labels/val/*'))

print(f'Train images : {len(train_imgs)}')
print(f'Val   images : {len(val_imgs)}')
print(f'Train labels : {len(train_lbls)}')
print(f'Val   labels : {len(val_lbls)}')

assert len(train_imgs) > 0, '❌ No training images found — check DATASET_DIR'
assert len(train_imgs) == len(train_lbls), '❌ Image/label count mismatch in train'
print('✅ Dataset looks good')

## 6 · Train

In [ ]:
from ultralytics import YOLO

# ── EDIT if needed ────────────────────────────────────────────────────────────
BASE_MODEL = 'yolov8n.pt'   # n=nano(fast) | s=small | m=medium | l=large
EPOCHS     = 50
IMG_SIZE   = 640
BATCH      = 16             # reduce to 8 if you get OOM errors
PROJECT    = '/content/drive/MyDrive/plate_runs'  # results saved here
RUN_NAME   = 'plate_v1'
# ─────────────────────────────────────────────────────────────────────────────

model = YOLO(BASE_MODEL)

results = model.train(
    data      = YAML_PATH,
    epochs    = EPOCHS,
    imgsz     = IMG_SIZE,
    batch     = BATCH,
    project   = PROJECT,
    name      = RUN_NAME,
    device    = 0,          # GPU
    patience  = 10,         # early stopping
    save      = True,
    plots     = True,
)

## 7 · Evaluate on validation set

In [ ]:
best_weights = os.path.join(PROJECT, RUN_NAME, 'weights/best.pt')
model_best   = YOLO(best_weights)
metrics      = model_best.val(data=YAML_PATH)

print(f"mAP@0.5      : {metrics.box.map50:.3f}")
print(f"mAP@0.5:0.95 : {metrics.box.map:.3f}")
print(f"Precision    : {metrics.box.mp:.3f}")
print(f"Recall       : {metrics.box.mr:.3f}")

## 8 · Quick visual test

In [ ]:
import random, cv2
from matplotlib import pyplot as plt

test_images = random.sample(val_imgs, min(4, len(val_imgs)))

fig, axes = plt.subplots(1, len(test_images), figsize=(16, 4))
if len(test_images) == 1:
    axes = [axes]

for ax, img_path in zip(axes, test_images):
    pred = model_best(img_path, conf=0.25, verbose=False)[0]
    annotated = pred.plot()  # BGR
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.axis('off')
    ax.set_title(os.path.basename(img_path))

plt.tight_layout()
plt.show()

## 9 · Copy `best.pt` to Drive root for easy download

In [ ]:
import shutil

dst = '/content/drive/MyDrive/best.pt'
shutil.copy(best_weights, dst)
print(f'✅ best.pt saved to {dst}')
print('Download it and replace best.pt in your FastAPI project.')